In [3]:
# pip install faiss-cpu

In [4]:
# faiss==> facebook ai search similarity library
import numpy as np
import faiss

In [5]:
chunks = [
    {"id": "C0", "text": "EMI bounce Rs500",          "vec": [0.9, 0.8]},
    {"id": "C1", "text": "Interest rate 13%",         "vec": [0.7, 0.2]},
    {"id": "C2", "text": "Prepayment after 6 EMIs",   "vec": [0.6, 0.7]},
    {"id": "C3", "text": "Gold loan LTV 75%",         "vec": [0.2, 0.1]},
    {"id": "C4", "text": "Pre-closing penalty 4%",    "vec": [0.5, 0.9]},
    {"id": "C5", "text": "Credit score 700+",         "vec": [0.1, 0.8]},
    {"id": "C6", "text": "EMI due date 5th of month", "vec": [0.8, 0.6]},
    {"id": "C7", "text": "Foreclosure after 12 EMIs", "vec": [0.7, 0.8]},
]


In [6]:
vectors=np.array([c["vec"] for c in chunks],dtype="float32")

In [7]:
dimension = vectors.shape[1]

flat_l2 = faiss.IndexFlatL2(dimension)
flat_l2.add(vectors)

In [8]:
print("total vectors:", flat_l2.ntotal)
memory_bytes = flat_l2.ntotal * dimension * 4
print("memory usage (bytes):", memory_bytes)

total vectors: 8
memory usage (bytes): 64


In [9]:
query_vector = np.array([[0.8, 0.7]], dtype="float32")
D, I = flat_l2.search(query_vector, k=3)
D,I

(array([[0.00999999, 0.02      , 0.02000001]], dtype=float32),
 array([[6, 0, 7]]))

In [10]:
I[0]

array([6, 0, 7])

In [11]:
chunks[6]['text']

'EMI due date 5th of month'

In [12]:
for rank, idx in enumerate(I[0]):
    print(f"Rank {rank+1}: Chunk ID: {chunks[idx]['id']}, Text: {chunks[idx]['text']}, Distance: {D[0][rank]:.4f}")

Rank 1: Chunk ID: C6, Text: EMI due date 5th of month, Distance: 0.0100
Rank 2: Chunk ID: C0, Text: EMI bounce Rs500, Distance: 0.0200
Rank 3: Chunk ID: C7, Text: Foreclosure after 12 EMIs, Distance: 0.0200


In [13]:
dimension = vectors.shape[1]

flat_l2 = faiss.IndexFlatIP(dimension)
flat_l2.add(vectors)


query_vector = np.array([[0.8, 0.7]], dtype="float32")
D, I = flat_l2.search(query_vector, k=3)
D,I


(array([[1.28     , 1.12     , 1.0600001]], dtype=float32), array([[0, 7, 6]]))

In [14]:
vectors_hnsw = np.array([c["vec"] for c in chunks], dtype="float32")
faiss.normalize_L2(vectors_hnsw)


M = 2
hnsw_index = faiss.IndexHNSWFlat(dimension, M)


# Build-time quality knob
hnsw_index.hnsw.efConstruction = 20

# Search-time quality knob
hnsw_index.hnsw.efSearch = 4





In [15]:
hnsw_index.add(vectors_hnsw)

In [16]:
query = "Emi due"
query_vector = np.array([[0.8, 0.7]], dtype="float32")
faiss.normalize_L2(query_vector)
D, I = hnsw_index.search(query_vector, k=3)
D,I


(array([[6.103254e-05, 5.671756e-03, 1.769912e-02]], dtype=float32),
 array([[0, 6, 7]]))

In [21]:
chunks = [
    {"id": 0, "text": "EMI bounce Rs500",         "vec": [0.9, 0.8]},
    {"id": 1, "text": "Interest rate 13%",         "vec": [0.7, 0.2]},
    {"id": 2, "text": "Prepayment after 6 EMIs",  "vec": [0.6, 0.7]},
    {"id": 3, "text": "Gold loan LTV 75%",         "vec": [0.2, 0.1]},
    {"id": 4, "text": "Pre-closing penalty 4%",   "vec": [0.5, 0.9]},
    {"id": 5, "text": "Credit score 700+",         "vec": [0.1, 0.8]},
]

vectors = np.array([c["vec"] for c in chunks], dtype="float32")


dim     = 2
nlist   = 2 


quantizer = faiss.IndexFlatL2(dim)

index     = faiss.IndexIVFFlat(quantizer, dim, nlist)

index.train(vectors)


WARNING clustering 6 points to 2 centroids: please provide at least 78 training points


In [22]:
index.add(vectors)

In [23]:
query  = np.array([[0.8, 0.75]], dtype="float32")
k      = 3

index.nprobe = 1 

D, I = index.search(query, k)

In [24]:
D,I

(array([[0.01249999, 0.0425    , 0.31250003]], dtype=float32),
 array([[0, 2, 1]]))